# 05 · Follow-up analyses (20 Nov 2026)

Answers the questions from the 18 Nov lab meeting. Scripts (run from the project root):
`09_pdgfra_check.py` → `05_DE_covariates.py <cell_type> <cov|nocov>` → `06_DA_subtypes.py` → `07_disease_composition.py` → `08_followup_figures.py`.
Steps 05, 06 and 09 need the count matrices, so they are not re-run here; this notebook reads their outputs and runs 07 and 08.

In [ ]:
import os, numpy as np, pandas as pd, statsmodels.formula.api as smf, warnings
from IPython.display import Image, display
warnings.filterwarnings("ignore")
if os.path.basename(os.getcwd()) == "notebooks": os.chdir("..")
pd.set_option("display.width", 160)

## 1 · Is PDGFRA an OPC or a fibroblast marker?
Pseudobulk CPM, all donors pooled. Fibroblasts express PDGFRA too, but the rest of each panel separates the two cleanly.

In [ ]:
c = pd.read_csv("results/followup/pdgfra_check_cpm.csv", index_col=0)
c.loc[["OPC","Fibro","Oligo","Astro","Mural","Endo","Micro_PVM"], ["PDGFRA","VCAN","OLIG2","SOX10","PTPRZ1","COL1A2","DCN","LAMA2","nuclei","donors"]]

In [ ]:
import runpy; _ = runpy.run_path("analysis/08_followup_figures.py")
display(Image("results/followup/figures/pdgfra_check.png", width=900))

## 2 · Pseudobulk DE with age, sex and PMI
Midbrain donors with all three covariates; the same donors are fitted with `~ dataset + condition` and with `~ dataset + age + sex + PMI + condition`.

In [ ]:
rows = []
for ct in ["Micro_PVM", "Astro", "Oligo", "OPC"]:
    a = pd.read_csv(f"results/followup/DE_cov_Midbrain_{ct}_nocov.csv"); b = pd.read_csv(f"results/followup/DE_cov_Midbrain_{ct}_cov.csv")
    sa, sb = set(a.gene[a.padj < .05]), set(b.gene[b.padj < .05])
    m = a.set_index("gene").log2FoldChange.to_frame("nocov").join(b.set_index("gene").log2FoldChange.rename("cov"))
    rows.append(dict(cell_type=ct, PD=a.n_PD[0], Ctrl=a.n_Ctrl[0], datasets=a.datasets[0], DE_without=len(sa), DE_with=len(sb), kept=len(sa & sb), lfc_correlation=round(m.corr().iloc[0, 1], 2)))
pd.DataFrame(rows)

In [ ]:
cov = pd.read_csv("results/followup/donor_covariates.csv")
meta = pd.read_csv("results/pseudobulk/pb_meta.csv")
mid = meta[meta.region.str.lower().str.contains("nigra|midbrain|snpc") & meta.diagnosis.isin(["PD", "Control"])].drop_duplicates("donor").merge(cov, on="donor")
mid.groupby("diagnosis")[["PMI", "age"]].median().round(1)   # PMI does not differ between groups

In [ ]:
display(Image("results/followup/figures/de_covariates.png", width=1000))
pd.read_csv("results/followup/watch_genes_cov_vs_nocov.csv").round(4)

## 3 · DA neuron subtypes (Kamath 2022: SOX6/AGTR1 DA neurons are the most vulnerable, CALB1 spared)

In [ ]:
d = pd.read_csv("results/followup/DA_subtype_per_donor.csv")
res = []
for mn in [5, 10]:
    s = d[(d.n >= mn) & d.diagnosis.isin(["PD", "Control"])].copy(); s["PD"] = (s.diagnosis == "PD").astype(int)
    s = s[s.dataset.isin([x for x, g in s.groupby("dataset") if g.PD.nunique() == 2])]
    for v in ["AGTR1p", "SOX6p", "CALB1p"]:
        s["y"] = np.log((s[v] + .5) / (s.n - s[v] + .5)); f = smf.ols("y ~ PD + C(dataset) + lumi", s).fit()
        res.append(dict(min_DA_nuclei=mn, donors=len(s), subtype=v, pct_Ctrl=round(100 * (s[v] / s.n)[s.PD == 0].mean(), 1),
                        pct_PD=round(100 * (s[v] / s.n)[s.PD == 1].mean(), 1), coef=round(f.params.PD, 2), p=round(f.pvalues.PD, 4)))
pd.DataFrame(res)

In [ ]:
display(Image("results/followup/figures/da_subtypes.png", width=1000))

## 4 · Other Lewy-body diagnoses vs control (composition)

In [ ]:
import importlib, sys; sys.path.insert(0, "analysis")
R = importlib.import_module("07_disease_composition").run()
R.sort_values("p").head(12).drop(columns="datasets").round(4)

**Reading.** PDD, DLB and LBD each come from one dataset with 4–6 donors, so nothing passes FDR. Cortical microglia rise in PD (3 datasets) and in PDD, which is worth checking in the full run.